# Grace in the Gap — Kaggle demo

**Scripture for the seconds when AI asks us to wait.** Grace in the Gap is a Claude Code plugin that turns AI latency — the pause while the model thinks or tests run — into an optional five‑second breath, a short pre‑authored reflection, and a verse of Scripture.

The production plugin is written in TypeScript. This notebook demonstrates the same pipeline in Python:

1. **On‑device logic (no keys):** privacy‑minimized event, deterministic selection over an approved catalog, and a 108‑scenario validation. This is exactly the on‑device fallback the live plugin uses when Gloo is unavailable.
2. **Live path (Gloo + YouVersion):** a real OAuth token, a real Gloo structured selection, and a real YouVersion passage + attribution fetch, composed into one card.

Grace is **live‑only** in production: there is no simulated content. The visible reflection always comes from the pre‑authored catalog and the verse always comes from YouVersion. Gloo only ever chooses approved IDs — it never writes devotional prose.

## 1. The approved catalog

Reflections are pre‑authored and review‑gated. Each profile maps a coarse task context to a tone, an approved reflection, and a Scripture reference (USFM).

In [ ]:
from hashlib import sha256
from itertools import product

PROFILES = [
    dict(id='quiet-trust', tasks=['generation','analysis','unknown'], tone='calm', snippet='One slow breath. The result can arrive without carrying all your attention.', usfm='PSA.46.10'),
    dict(id='purposeful-work', tasks=['implementation','refactor'], tone='steady', snippet='Let the build run. Return to the purpose beneath the task.', usfm='COL.3.23'),
    dict(id='wisdom-in-debugging', tasks=['debugging'], tone='reflective', snippet='The bug can wait five seconds. Make room for wisdom before the next attempt.', usfm='JAS.1.5'),
    dict(id='patient-in-testing', tasks=['testing'], tone='encouraging', snippet='While the tests run, loosen your shoulders and hold on to hope.', usfm='ROM.12.12'),
    dict(id='committed-plans', tasks=['planning'], tone='steady', snippet='Commit the work. Hold the outcome with open hands.', usfm='PRO.16.3'),
    dict(id='thoughtful-review', tasks=['review'], tone='reflective', snippet='Before finding the flaw, take a moment to notice what is true and good.', usfm='PHP.4.8'),
    dict(id='rest-for-late-work', tasks=['generation','implementation','debugging','testing','analysis','refactor','planning','review','unknown'], tone='calm', snippet='This task is not a measure of your worth. Receive a moment of rest.', usfm='MAT.11.28', windows=['late-evening']),
    dict(id='care-in-uncertainty', tasks=['debugging','analysis','unknown'], tone='encouraging', snippet='Name the worry, then set it down for the length of one breath.', usfm='1PE.5.7'),
]

REFERENCES = {
    'PSA.46.10': 'Psalm 46:10', 'COL.3.23': 'Colossians 3:23', 'JAS.1.5': 'James 1:5',
    'ROM.12.12': 'Romans 12:12', 'PRO.16.3': 'Proverbs 16:3', 'PHP.4.8': 'Philippians 4:8',
    'MAT.11.28': 'Matthew 11:28', '1PE.5.7': '1 Peter 5:7',
}
print(f'{len(PROFILES)} approved profiles, {len(REFERENCES)} passage references')

## 2. Privacy‑minimized event and on‑device selection (no keys)

The plugin never sends the prompt, code, filenames, or paths anywhere. It derives a coarse task label on device and hashes the session id. The same deterministic selector below is what the live plugin falls back to if Gloo is unreachable.

In [ ]:
def safe_event(task_type, duration_bucket='8-15', time_window='afternoon', locale='en-US', session='demo'):
    # The complete provider-safe payload: no prompt, code, path, or transcript.
    return dict(surface='claude-code', taskType=task_type, durationBucket=duration_bucket,
                timeWindow=time_window, locale=locale,
                sessionHash=sha256(session.encode()).hexdigest()[:24])

def candidates_for(event):
    return [p for p in PROFILES if event['taskType'] in p['tasks']
            and ('windows' not in p or event['timeWindow'] in p['windows'])]

def select_locally(event):
    eligible = candidates_for(event)
    assert eligible, 'Every supported event needs a candidate profile'
    # Weight-free, deterministic pick keyed on coarse labels only.
    seed = f"{event['sessionHash']}:{event['taskType']}:{event['timeWindow']}"
    return eligible[int(sha256(seed.encode()).hexdigest()[:8], 16) % len(eligible)]

event = safe_event('debugging', session='kaggle-demo')
profile = select_locally(event)
print('provider-safe event:', event)
print('selected reflection:', profile['snippet'])
print('reference:', REFERENCES[profile['usfm']])

## 3. Deterministic evaluation (108 scenarios)

A contest build needs more than one hero example. This checks every task × time‑window × duration combination for a valid, catalog‑resolvable choice and wide passage diversity.

In [ ]:
tasks = ['generation','implementation','debugging','testing','analysis','refactor','planning','review','unknown']
windows = ['morning','afternoon','evening','late-evening']
buckets = ['8-15','16-30','over-30']
distribution, failures = {}, []
for index, (task, window, bucket) in enumerate(product(tasks, windows, buckets)):
    try:
        ev = safe_event(task, bucket, window, session=f'eval-{index}')
        pr = select_locally(ev)
        assert pr['snippet'] and REFERENCES[pr['usfm']]
        distribution[pr['usfm']] = distribution.get(pr['usfm'], 0) + 1
    except Exception as exc:
        failures.append((task, window, bucket, str(exc)))
print({'scenarios': len(tasks)*len(windows)*len(buckets), 'passed': (len(tasks)*len(windows)*len(buckets))-len(failures),
       'failures': len(failures), 'unique_passages': len(distribution), 'distribution': distribution})
assert not failures and len(distribution) >= 6

## 4. Live path — Gloo + YouVersion

This section makes real API calls. Provide credentials via environment variables or Kaggle Secrets (`GLOO_CLIENT_ID`, `GLOO_CLIENT_SECRET`, `YVP_APP_KEY`). Both APIs are free for challenge participants. Without credentials, the cell prints setup instructions and stops — it never fabricates Scripture.

In [ ]:
import os, json, base64, requests

def get_credentials():
    creds = {k: os.environ.get(k, '') for k in ('GLOO_CLIENT_ID', 'GLOO_CLIENT_SECRET', 'YVP_APP_KEY')}
    if not all(creds.values()):
        try:  # Kaggle Secrets, if configured
            from kaggle_secrets import UserSecretsClient
            client = UserSecretsClient()
            for key in creds:
                if not creds[key]:
                    try: creds[key] = client.get_secret(key)
                    except Exception: pass
        except Exception:
            pass
    return creds

GLOO_BASE = os.environ.get('GLOO_BASE_URL', 'https://platform.ai.gloo.com')
YV_BASE = os.environ.get('YOUVERSION_BASE_URL', 'https://api.youversion.com')
GLOO_MODEL = os.environ.get('GLOO_MODEL', 'gloo-openai-gpt-5-mini')
VERSION_ID = os.environ.get('GRACE_BIBLE_VERSION_ID', '3034')  # Berean Standard Bible (public domain)
creds = get_credentials()
HAVE_KEYS = all(creds.values())
print('Credentials detected.' if HAVE_KEYS else 'No credentials — the live cell below will print setup steps and skip.')

In [ ]:
def gloo_token(client_id, client_secret):
    auth = base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode()
    r = requests.post(f'{GLOO_BASE}/oauth2/token',
                      headers={'Authorization': f'Basic {auth}', 'Content-Type': 'application/x-www-form-urlencoded'},
                      data={'grant_type': 'client_credentials', 'scope': 'api/access'}, timeout=15)
    r.raise_for_status()
    return r.json()['access_token']

def gloo_select(token, event):
    allowed = [dict(momentProfileId=p['id'], reflectionSnippetId=p['snippet'], passageHint=p['usfm'], tone=p['tone'])
               for p in candidates_for(event)]
    instructions = ('You are a structured selector for Grace in the Gap. Return one JSON object only, no prose. '
                    'Choose one allowed candidate without changing any field. Use durationSeconds 5 or 8, '
                    'confidence 0..1, fallbackVotd false, needsAuth false. '
                    f'Allowed candidates: {json.dumps(allowed)}. '
                    'Keys: momentProfileId, reflectionSnippetId, passageHint, durationSeconds, tone, confidence, fallbackVotd, needsAuth.')
    r = requests.post(f'{GLOO_BASE}/ai/v1/responses',
                      headers={'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'},
                      json={'model': GLOO_MODEL, 'instructions': instructions,
                            'input': [{'role': 'user', 'content': json.dumps(event)}]}, timeout=30)
    r.raise_for_status()
    message = next(item for item in r.json()['output'] if item['type'] == 'message')
    return json.loads(message['content'][0]['text'])

def valid_choice(decision):
    if decision.get('confidence', 0) < 0.55 or decision.get('needsAuth') or decision.get('fallbackVotd'):
        return False
    p = next((p for p in PROFILES if p['id'] == decision.get('momentProfileId')), None)
    return bool(p and p['snippet'] == decision.get('reflectionSnippetId')
                and p['usfm'] == decision.get('passageHint') and p['tone'] == decision.get('tone'))

def yv_passage(app_key, usfm, locale='en-US'):
    headers = {'X-YVP-App-Key': app_key, 'Accept-Language': locale, 'Accept': 'application/json'}
    p = requests.get(f'{YV_BASE}/v1/bibles/{VERSION_ID}/passages/{usfm}', params={'format': 'text'}, headers=headers, timeout=15)
    p.raise_for_status()
    m = requests.get(f'{YV_BASE}/v1/bibles/{VERSION_ID}', headers=headers, timeout=15)
    m.raise_for_status()
    pj, mj = p.json(), m.json()
    copyright = (mj.get('copyright') or mj.get('promotional_content') or '').strip().strip('"')
    return dict(text=pj['content'], reference=pj.get('reference', usfm),
                version=mj.get('abbreviation') or mj.get('title') or f'Bible {VERSION_ID}', copyright=copyright)

if not HAVE_KEYS:
    print('Set GLOO_CLIENT_ID, GLOO_CLIENT_SECRET, YVP_APP_KEY (env vars or Kaggle Secrets) to run the live card.')
    print('  Gloo:       https://studio.ai.gloo.com  > API Credentials')
    print('  YouVersion: https://developers.youversion.com  > App Key')
else:
    live_event = safe_event('debugging', session='kaggle-live-demo')
    token = gloo_token(creds['GLOO_CLIENT_ID'], creds['GLOO_CLIENT_SECRET'])
    decision = gloo_select(token, live_event)
    degraded = not valid_choice(decision)
    profile = next(p for p in PROFILES if p['id'] == decision['momentProfileId']) if not degraded else select_locally(live_event)
    passage = yv_passage(creds['YVP_APP_KEY'], profile['usfm'], live_event['locale'])
    print('=' * 68)
    print(f"Grace in the Gap · 5s pause · GLOO + YOUVERSION{'  (selector degraded)' if degraded else ''}")
    print()
    print(f"\"{passage['text']}\"")
    print(f"{passage['reference']} · {passage['version']}")
    print()
    print(profile['snippet'])
    print(f"Attribution: {passage['copyright']}")
    print('Privacy: raw prompt is neither stored nor transmitted.')
    print('=' * 68)

## Verification boundary

This notebook mirrors the production TypeScript plugin. The public repository is the source of truth for the OAuth token broker, the strict Gloo response validation, the YouVersion adapter (passage text plus a separate Bible‑metadata fetch for required copyright), the async Claude Code hook, the MCP server, and the automated test suite.